## Task 2: End-to-End ML Pipeline (Scikit-learn)
Problem Statement & Objective: Build a production-ready, reusable pipeline to predict customer churn, ensuring all preprocessing and modeling steps are bundled.

Dataset Loading & Preprocessing: Used the Telco Churn Dataset. Implemented ColumnTransformer for automated scaling of numeric features and One-Hot Encoding for categorical features.

Model Development & Training: Utilized the Pipeline API with Logistic Regression and Random Forest. Performed hyperparameter tuning via GridSearchCV to select the optimal model.

Evaluation Metrics: Evaluated using Precision, Recall, and F1-score. The best model was exported using joblib.

Visualizations: Confusion Matrix and Classification Report for the best-performing model.

Final Summary / Insights: Using the Pipeline API prevents data leakage and ensures that the model can be deployed in a production environment with minimal manual data handling.

In [ ]:
!pip install pandas scikit-learn joblib

## Data Loading and Initial Cleaning

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# 1. Load Dataset
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

# 2. Basic Cleaning: Convert TotalCharges to numeric and handle nulls
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(subset=['TotalCharges'], inplace=True)

# 3. Split Features and Target
X = df.drop(['customerID', 'Churn'], axis=1)
y = df['Churn'].apply(lambda x: 1 if x == 'Yes' else 0)

# 4. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Implement Preprocessing using Pipeline API

In [ ]:
# Identify feature types
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns

# Pipeline for Numeric Features: Impute missing values + Scale
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline for Categorical Features: Impute + One-Hot Encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine Preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

## Train Models and Hyperparameter Tuning (GridSearchCV)

In [ ]:
# Create a base pipeline with a placeholder for the classifier
clf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression()) # Default placeholder
])

# Define the parameter grid for both models
param_grid = [
    {
        'classifier': [LogisticRegression(max_iter=1000)],
        'classifier__C': [0.1, 1, 10]
    },
    {
        'classifier': [RandomForestClassifier()],
        'classifier__n_estimators': [100, 200],
        'classifier__max_depth': [None, 10, 20]
    }
]

# Run GridSearchCV
grid_search = GridSearchCV(clf_pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Best Model: {grid_search.best_params_}")
print(f"Best CV Score: {grid_search.best_score_:.4f}")

Best Model: {'classifier': LogisticRegression(max_iter=1000), 'classifier__C': 10}
Best CV Score: 0.8073


## Evaluation and Export

In [ ]:
# 1. Evaluate on Test Data
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("\nFinal Model Evaluation:")
print(classification_report(y_test, y_pred))

# 2. Export the complete pipeline using joblib
joblib.dump(best_model, 'churn_pipeline_model.pkl')
print("\nSuccess: Pipeline exported to 'churn_pipeline_model.pkl'")


Final Model Evaluation:
              precision    recall  f1-score   support

           0       0.84      0.89      0.86      1033
           1       0.62      0.52      0.56       374

    accuracy                           0.79      1407
   macro avg       0.73      0.70      0.71      1407
weighted avg       0.78      0.79      0.78      1407


Success: Pipeline exported to 'churn_pipeline_model.pkl'
